In [ ]:
import pandas as pd
import tensorflow as tf

## Leitura dos dados

In [ ]:
########################################
# Download dos dados
########################################
# Extrai os arquivos para 
# a pasta /content/dataset
########################################

%%bash --out output --err error
set -euxo pipefail

if [ ! -d dataset ]; then
    mkdir -p "/root/.kaggle"
    cp -a "/content/drive/My Drive/<PASTA_DO_PROJETO>/kaggle.json" "/root/.kaggle"
    chmod 600 "/root/.kaggle/kaggle.json"

    pip install kaggle==1.5.6
    kaggle datasets download -d "paultimothymooney/kermany2018" -p "/content"

    unzip "/content/kermany2018.zip" "OCT2017 /*" -d "/content"
    mv "OCT2017 " "dataset"
fi

In [ ]:
########################################
# Leitura dos dados
########################################
# split_paper.csv
# ---
# Divide as bases
# de acordo com o artigo,
# logo unindo validação ao teste.
# Sem exclusão de pacientes duplicados.
########################################
# split_baseline.csv
# ---
# Divide as imagens
# unindo validação ao treino,
# sem excluir pacientes contidos
# no conjunto de (treino ∪ val) ∩ teste.
########################################
# split_honest.csv
# ---
# Divide os dados
# unindo validação ao treino,
# excluindo pacientes contidos
# no conjunto de (treino ∪ val) ∩ teste.
# É a que serve como comparação 
# direta com o outro grupo de OCT.
########################################

df = pd.read_csv("/content/drive/My Drive/<PASTA_DO_PROJETO>/Notebooks_Vinicius/split_honest.csv")

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1.0/255,
)

train_generator = datagen.flow_from_dataframe(
    df.query('new_split == "train"'),
    x_col='path',
    y_col='file_label',
    class_mode='categorical',
)

val_generator = datagen.flow_from_dataframe(
    df.query('new_split == "val"'),
    x_col='path',
    y_col='file_label',
    class_mode='categorical',
)

test_generator = datagen.flow_from_dataframe(
    df.query('new_split == "test"'),
    x_col='path',
    y_col='file_label',
    class_mode='categorical',
)

## Códigos de divisão da base

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import StratifiedShuffleSplit

In [ ]:
########################################
# Lista as imagens
########################################
# Cria um dataframe com 
# os metadados das imagens
########################################

def list_files():
    imgpaths = Path("dataset").glob("**/*")
    imgpaths = filter(lambda x: str(x).endswith("jpeg"), imgpaths)
    imgpaths = list(imgpaths)

    df = pd.DataFrame({
        'path': imgpaths,
        'image_n': list(map(lambda x: x.parts[3].split("-")[2][0], imgpaths)),
        'patient_id': list(map(lambda x: x.parts[3].split("-")[1], imgpaths)),
        'file_label': list(map(lambda x: x.parts[3].split("-")[0], imgpaths)),
        'folder_label': list(map(lambda x: x.parts[2], imgpaths)),
        'old_split': list(map(lambda x: x.parts[1], imgpaths)),
    })

    return df

In [ ]:
########################################
# Divide as imagens
########################################
# De acordo com o artigo,
# logo unindo validação ao teste.
# Sem exclusão de pacientes duplicados.
########################################

def split_paper():
    df = list_files()
    train_df = df.query('old_split == "train"')

    sss = StratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)
    split = sss.split(train_df.index.values, train_df.file_label.values)
    train_idx, val_idx = list(split)[0]

    train_idx = train_df.index.values[train_idx]
    val_idx = train_df.index.values[val_idx]

    new_split = np.array(['test'] * len(df), dtype=object)
    new_split[train_idx] = 'train'
    new_split[val_idx] = 'val'

    df['new_split'] = new_split
    df['path'] = df['path'].astype('str')

    return df

# df = split_paper()
# df.to_csv("drive/MyDrive/Retinal_OCT_Images/Base/split_paper.csv")

In [ ]:
########################################
# Divide as imagens
########################################
# Unindo validação ao treino,
# sem excluir pacientes contidos
# no conjunto de (treino ∪ val) ∩ teste.
########################################

def split_baseline():
    df = list_files()
    train_df = df.query('old_split == "train" | old_split == "val"')

    sss = StratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)
    split = sss.split(train_df.index.values, train_df.file_label.values)
    train_idx, val_idx = list(split)[0]

    train_idx = train_df.index.values[train_idx]
    val_idx = train_df.index.values[val_idx]

    new_split = np.array(['test'] * len(df), dtype=object)
    new_split[train_idx] = 'train'
    new_split[val_idx] = 'val'

    df['new_split'] = new_split
    df['path'] = df['path'].astype('str')

    return df

# df = split_baseline()
# df.to_csv("drive/MyDrive/Retinal_OCT_Images/Base/split_baseline.csv")

In [ ]:
########################################
# Divide as imagens
########################################
# Unindo validação ao treino,
# exclundo pacientes contidos
# no conjunto de (treino ∪ val) ∩ teste.
########################################
# É o que serve de comparação 
# direta com o outro grupo de OCT.
########################################

def split_honest():
    df = list_files()
    test_patients = df.query('old_split == "test"').patient_id.values
    df = df.query('patient_id not in @test_patients | old_split == "test"')
    df = df.reset_index(drop=True)

    train_df = df.query('old_split == "train" | old_split == "val"')

    gss = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
    split = gss.split(
        train_df.index.values,
        train_df.file_label.values,
        train_df.patient_id.values,
    )
    train_idx, val_idx = list(split)[0]

    train_idx = train_df.index.values[train_idx]
    val_idx = train_df.index.values[val_idx]

    new_split = np.array(['test'] * len(df), dtype=object)
    new_split[train_idx] = 'train'
    new_split[val_idx] = 'val'

    df['new_split'] = new_split
    df['path'] = df['path'].astype('str')

    return df

# df = split_honest()
# df.to_csv("drive/MyDrive/Retinal_OCT_Images/Base/split_honest.csv")

## Testa os códigos de divisão da base

In [ ]:
def test_split_paper():
    df = split_paper()

    # testa se o treino não é teste
    assert (df.query('old_split == "train"').new_split != "test").all()

    # testa se a validação é teste
    assert (df.query('old_split == "val"').new_split == 'test').all()

    # testa se o teste é teste
    assert (df.query('old_split == "test"').new_split == 'test').all()

test_split_paper()

In [ ]:
def test_split_baseline():
    df = split_baseline()

    # testa se o treino não é teste
    assert (df.query('old_split == "train"').new_split != "test").all()

    # testa se a validação não é teste
    assert (df.query('old_split == "val"').new_split != 'test').all()

    # testa se o teste é teste
    assert (df.query('old_split == "test"').new_split == 'test').all()

test_split_baseline()

In [ ]:
def test_split_honest():
    df = split_honest()

    # testa se o treino não é teste
    assert (df.query('old_split == "train"').new_split != "test").all()

    # testa se a validação não é teste
    assert (df.query('old_split == "val"').new_split != 'test').all()

    # testa se o teste é teste
    assert (df.query('old_split == "test"').new_split == 'test').all()

    # teste se o treino tem pacientes da validação
    train_patients = df.query('new_split == "train"').patient_id.values
    val_patients = df.query('new_split == "val"').patient_id.values
    assert not set(train_patients).intersection(val_patients)

    # teste se o treino tem pacientes do teste
    test_patients = df.query('new_split == "test"').patient_id.values
    assert not set(train_patients).intersection(test_patients)

    # teste se a validação tem pacientes do teste
    assert not set(val_patients).intersection(test_patients)

test_split_honest()

In [ ]:
def test_split_honest_is_stratified():
    df = pd.read_csv("drive/MyDrive/Retinal_OCT_Images/Base/split_honest.csv")

    train = df.query('new_split == "train"').file_label.value_counts()
    print("Divisão do treino\n---\n", train / train.sum(), sep="")

    print()

    val = df.query('new_split == "val"').file_label.value_counts()
    print("Divisão da validação\n---\n", val / val.sum(), sep="")